# Toronto Café Location-Scoring: Transit Stop Ingest

**Pipeline stage:** Bronze (raw landing)  
**Source:** Toronto Transit Commission (TTC) GTFS feed, via the City of Toronto CKAN API  
**Source URL:** https://open.toronto.ca/dataset/ttc-routes-and-schedules/  
**Destination:** MongoDB `toronto_cafe.transit_stops`

This notebook pulls every TTC stop (subway, streetcar, and bus) from the official GTFS transit feed and lands them in the database as the raw bronze layer. These stops are the **foot-traffic layer** of the scoring model: proximity to transit is a positive factor, since nearby stops bring people past a location.

The feed is fetched through the open-data API rather than a manual download, so re-running this notebook always pulls the current feed. Each stop becomes one document keyed on its GTFS `stop_id`, and records where it came from and when it was pulled.

## 1. Setup and connection

Import the libraries, load the connection string from the local `.env`, and connect to the project database. This is the same database as the other collections; the stops land in a new collection called `transit_stops`.

In [1]:
import os
import requests
import zipfile, io                             # unzip the GTFS file in memory
import pandas as pd                            # read stops.txt (a CSV)
from datetime import datetime, timezone
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv(".env")                            # load MONGODB_URI from .env
client = MongoClient(os.environ["MONGODB_URI"])   # connect to the same Atlas cluster
db = client["toronto_cafe"]                    # same project database as before

C:\Users\moham\anaconda3\Lib\site-packages\pymongo\pyopenssl_context.py:348: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  _crypto.X509.from_cryptography(x509.load_der_x509_certificate(cert))


## 2. Fetch the GTFS feed from the open-data API

Toronto's open-data portal runs on **CKAN**. Instead of hardcoding a download link, we ask CKAN for the dataset's file list, find the GTFS zip, download it, unzip it **in memory**, and read `stops.txt` (a CSV) into a table. Because we ask the API each time, re-running always fetches the latest feed.

In [2]:
# 1. Ask Toronto's open-data API (CKAN) for the dataset info
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
package = requests.get(
    base_url + "/api/3/action/package_show",
    params={"id": "ttc-routes-and-schedules"}
).json()

# 2. Find the GTFS zip file in the dataset
zip_url = None
for resource in package["result"]["resources"]:   # look through each file in the dataset
    if resource["format"] == "ZIP":               # the GTFS feed is the ZIP one
        zip_url = resource["url"]

# 3. Download the zip
response = requests.get(zip_url)
response.raise_for_status()                        # stop with a clear error if the download failed

# 4. Unzip in memory and read stops.txt into a table
zf = zipfile.ZipFile(io.BytesIO(response.content)) # treat the downloaded bytes as a file
stops = pd.read_csv(zf.open("stops.txt"))          # read the stops CSV
print("stops loaded:", len(stops))

stops loaded: 9361


## 3. Inspect the stops

Look at the size and columns of the stops table before landing anything. The key columns are `stop_id` (the unique key), `stop_name`, `stop_lat`, and `stop_lon`.

In [3]:
print("shape:", stops.shape)                   # (rows, columns)
print("columns:", stops.columns.tolist())      # the column names
stops.head()                                   # first few stops

shape: (9361, 12)
columns: ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station', 'stop_timezone', 'wheelchair_boarding']


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,662,662,Danforth Rd at Kennedy Rd,NaN,43.714379,-79.260939,NaN,NaN,NaN,NaN,NaN,1
1,929,929,Davenport Rd at Bedford Rd,NaN,43.674448,-79.399659,NaN,NaN,NaN,NaN,NaN,1
2,940,940,Davenport Rd at Dupont St,NaN,43.675511,-79.401938,NaN,NaN,NaN,NaN,NaN,2
3,1871,1871,Davisville Ave at Cleveland St,NaN,43.702088,-79.378112,NaN,NaN,NaN,NaN,NaN,1
4,11700,11700,Disco Rd at Attwell Dr,NaN,43.701362,-79.594843,NaN,NaN,NaN,NaN,NaN,1


## 4. Reshape and land into the bronze layer

Walk through every stop, build a document keyed on its `stop_id`, record its name and position, and stamp it with source and pull time. Same reshape-and-upsert pattern as the other two collections. The numbers come out of the file as pandas number types, so they are converted to plain `int` / `float` values that the database can store.

In [4]:
transit_stops = db["transit_stops"]                # destination collection

for row in stops.to_dict(orient="records"):        # each stop as a dictionary
    doc = {
        "_id": int(row["stop_id"]),                # stop_id as the key (plain int)
        "stop_name": row["stop_name"],             # the stop's name
        "lat": float(row["stop_lat"]),             # position (plain float)
        "lon": float(row["stop_lon"]),             # position (plain float)
        "source": "ttc_gtfs",                      # where it came from
        "ingested_at": datetime.now(timezone.utc), # when we pulled it
    }
    transit_stops.update_one(                      # upsert: update if it exists, insert if new
        {"_id": doc["_id"]},
        {"$set": doc},
        upsert=True,
    )

print("done — landed", len(stops), "stops")

done — landed 9361 stops


## 5. Verify the load

Confirm the stops landed and check the shape of one.

In [5]:
print(transit_stops.count_documents({}))       # how many stops landed
transit_stops.find_one()                        # peek at one landed stop

9361


{'_id': 662,
 'ingested_at': datetime.datetime(2026, 7, 24, 5, 19, 40, 281000),
 'lat': 43.714379,
 'lon': -79.260939,
 'source': 'ttc_gtfs',
 'stop_name': 'Danforth Rd at Kennedy Rd'}